In [0]:
from pyspark.sql import functions as fn

def working_days_calendar(spark):
    """
    Create a calendar DataFrame with monthly working days.
    Covers 10 years from 2020-01-01.
    Excludes weekends as non-working days.
    Prints sample data for debugging.
    """
    
    df = spark.range(0, 365 * 10).select(
        fn.expr("date_add('2020-01-01', cast(id as int))").alias("Date")
    ).withColumn(
        "Month_Start", fn.trunc("Date", "MM")
    ).withColumn(
        "Working_Day_Type",
        fn.when(fn.date_format("Date", "EEEE").isin("Saturday", "Sunday"), "N").otherwise("Y")
    ).withColumn(
        "Working_Day_Calc",
        fn.when(fn.col("Working_Day_Type") == "Y", 1).otherwise(0)
    )

    # DEBUG: Print the schema and a small sample of data
    print("\n=== Calendar Schema ===")
    df.printSchema()
    print("\n=== Calendar Sample Data ===")
    df.orderBy("Date").show(12, truncate=False)

    # Optional: Aggregate by month to get total working days
    cal_agg = df.groupBy("Month_Start").agg(
        fn.sum("Working_Day_Calc").alias("Total_Working_Days")
    )

    # DEBUG: Print aggregated result
    print("\n=== Aggregated Calendar Sample ===")
    cal_agg.orderBy("Month_Start").show(12, truncate=False)

    return cal_agg

calendar_df = working_days_calendar(spark)
